The object that we wish to compute: $D_m \bar{P}_n D_r P_s$

In [1]:
import sympy as sp

m,n,r,s = sp.symbols('m n r s', real=True)
tau1 = sp.Function('tau_1', real=True)(m, n, r, s)
tau2 = sp.Function('tau_2', real=True, positive=True)(m, n, r, s)
tau = tau1 + sp.I*tau2
taub = tau1 - sp.I*tau2

In [2]:
P_s = sp.Rational(1, 2) * sp.I / tau2 * sp.diff(tau, s)
Pb_n = -sp.Rational(1, 2) * sp.I / tau2 * sp.diff(taub, n)
Q_m = -sp.Rational(1, 2) / tau2 * sp.diff(tau1, m)
Q_r = -sp.Rational(1, 2) / tau2 * sp.diff(tau1, r)

In [3]:
D_m_Pb_n = sp.diff(Pb_n, m) + 2 * sp.I * Q_m * Pb_n
D_m_Pb_n = sp.expand(sp.simplify(D_m_Pb_n))
D_r_P_s = sp.diff(P_s, r) - 2 * sp.I * Q_r * P_s
D_r_P_s = sp.expand(sp.simplify(D_r_P_s))

In [4]:
D_r_P_s

I*Derivative(tau_1(m, n, r, s), r, s)/(2*tau_2(m, n, r, s)) - Derivative(tau_2(m, n, r, s), r, s)/(2*tau_2(m, n, r, s)) - Derivative(tau_1(m, n, r, s), r)*Derivative(tau_1(m, n, r, s), s)/(2*tau_2(m, n, r, s)**2) - I*Derivative(tau_1(m, n, r, s), r)*Derivative(tau_2(m, n, r, s), s)/(2*tau_2(m, n, r, s)**2) - I*Derivative(tau_1(m, n, r, s), s)*Derivative(tau_2(m, n, r, s), r)/(2*tau_2(m, n, r, s)**2) + Derivative(tau_2(m, n, r, s), r)*Derivative(tau_2(m, n, r, s), s)/(2*tau_2(m, n, r, s)**2)

In [5]:
from itertools import product as iterprod

# The functions and coordinates to symbolify
coords = [m, n, r, s]
coord_names = {m: 'm', n: 'n', r: 'r', s: 's'}
funcs = [tau1, tau2]
func_names = {tau1: r'\tau_1', tau2: r'\tau_2'}

# Function symbols (tau1 -> \tau_1, etc.)
T_func_syms = {}
T_func_subs = []
for f in funcs:
    sym = sp.Symbol(func_names[f])
    T_func_syms[f] = sym
    T_func_subs.append((f, sym))

# First derivative symbols: nabla_i f
T_d1_syms = {}
T_deriv1_subs = []
for f, xi in iterprod(funcs, coords):
    name = rf'\nabla_{{{coord_names[xi]}}} {func_names[f]}'
    sym = sp.Symbol(name)
    T_d1_syms[(f, xi)] = sym
    T_deriv1_subs.append((sp.diff(f, xi), sym))

# Second derivative symbols: nabla_i nabla_j f
T_d2_syms = {}
T_deriv2_subs = []
for f, (xi, xj) in iterprod(funcs, iterprod(coords, coords)):
    deriv = sp.diff(f, xi, xj)
    if deriv == 0:
        continue
    # Use sorted key so nabla_m nabla_n = nabla_n nabla_m map to same symbol
    key = (f, tuple(sorted([xi, xj], key=lambda x: coord_names[x])))
    if key not in T_d2_syms:
        ci, cj = coord_names[key[1][0]], coord_names[key[1][1]]
        name = rf'\nabla_{{{ci}}} \nabla_{{{cj}}} {func_names[f]}'
        T_d2_syms[key] = sp.Symbol(name)
    T_deriv2_subs.append((deriv, T_d2_syms[key]))

print(f"Function subs: {len(T_func_subs)}")
print(f"1st deriv subs: {len(T_deriv1_subs)}")
print(f"2nd deriv subs: {len(T_deriv2_subs)}")

Function subs: 2
1st deriv subs: 8
2nd deriv subs: 32


In [6]:
# Apply substitutions: second derivs -> first derivs -> functions
D_r_P_s = D_r_P_s.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs)
D_m_Pb_n = D_m_Pb_n.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs)
D_r_P_s

I*\nabla_{r} \nabla_{s} \tau_1/(2*\tau_2) - \nabla_{r} \nabla_{s} \tau_2/(2*\tau_2) - \nabla_{r} \tau_1*\nabla_{s} \tau_1/(2*\tau_2**2) - I*\nabla_{r} \tau_1*\nabla_{s} \tau_2/(2*\tau_2**2) - I*\nabla_{r} \tau_2*\nabla_{s} \tau_1/(2*\tau_2**2) + \nabla_{r} \tau_2*\nabla_{s} \tau_2/(2*\tau_2**2)

We find that
$$
D_r P_s = A + iB
$$
where
$$
A = \frac{\nabla_r \tau_"\nabla_s\tau_2}{2\tau_2^2} - \frac{\nabla_r\nabla_s\tau_2}{2\tau_2} - \frac{\nabla_r\tau_1\nabla_s\tau_1}{2\tau_2^2}
$$
$$\boxed{A = -\tau_2, R_{1r1s} + \tfrac{1}{2},g^{ab}R_{arbs}}$$
and
$$
B = \frac{\nabla_r\nabla_s\tau_1}{2\tau_2} - \frac{\nabla_r\tau_1\nabla_s\tau_2}{2\tau_2^2} - \frac{\nabla_r\tau_2\nabla_s\tau_1}{2\tau_2^2}
$$
$$\boxed{B = \tau_1, R_{1r1s} - \tfrac{1}{2}(R_{1r2s}+R_{2r1s})}$$

In [7]:
# Define Riemann tensor component symbols
tau1_sym = T_func_syms[tau1]  # 	au_1
tau2_sym = T_func_syms[tau2]  # 	au_2

# R_{1m1n}, R_{1m2n}, R_{2m1n}, R_{2m2n} (and same with r,s)
R_1m1n = sp.Symbol(r'R_{1m1n}')
R_1m2n = sp.Symbol(r'R_{1m2n}')
R_2m1n = sp.Symbol(r'R_{2m1n}')
R_2m2n = sp.Symbol(r'R_{2m2n}')
# R_trace_mn = sp.Symbol(r'g^{ab}R_{ambn}')  # trace
R_trace_mn = (tau1_sym**2 + tau2_sym**2)/tau2_sym * R_1m1n - tau1_sym / tau2_sym * (R_1m2n + R_2m1n) + 1/tau2_sym * R_2m2n

R_1r1s = sp.Symbol(r'R_{1r1s}')
R_1r2s = sp.Symbol(r'R_{1r2s}')
R_2r1s = sp.Symbol(r'R_{2r1s}')
R_2r2s = sp.Symbol(r'R_{2r2s}')
# R_trace_rs = sp.Symbol(r'g^{ab}R_{arbs}')  # trace
R_trace_rs = (tau1_sym**2 + tau2_sym**2)/tau2_sym * R_1r1s - tau1_sym / tau2_sym * (R_1r2s + R_2r1s) + 1/tau2_sym * R_2r2s

# D_r P_s = A_rs + i B_rs where:
# A_rs = -tau2 R_{1r1s} + (1/2) g^{ab} R_{arbs}
# B_rs = tau1 R_{1r1s} - (1/2)(R_{1r2s} + R_{2r1s})
A_rs = -tau2_sym * R_1r1s + sp.Rational(1,2) * R_trace_rs
B_rs = tau1_sym * R_1r1s - sp.Rational(1,2) * (R_1r2s + R_2r1s)

# D_m Pb_n = conj(D_m P_n) -> Re same, Im flipped
# D_m Pb_n = A_mn - i B_mn  (complex conjugate of D_m P_n)
A_mn = -tau2_sym * R_1m1n + sp.Rational(1,2) * R_trace_mn
B_mn = tau1_sym * R_1m1n - sp.Rational(1,2) * (R_1m2n + R_2m1n)

print('D_r P_s = A_rs + i B_rs')
print('A_rs =', A_rs)
print('B_rs =', B_rs)
print()
print('D_m Pb_n = A_mn - i B_mn')
print('A_mn =', A_mn)
print('B_mn =', B_mn)


D_r P_s = A_rs + i B_rs
A_rs = -R_{1r1s}*\tau_2 + R_{1r1s}*(\tau_1**2 + \tau_2**2)/(2*\tau_2) + R_{2r2s}/(2*\tau_2) - \tau_1*(R_{1r2s} + R_{2r1s})/(2*\tau_2)
B_rs = R_{1r1s}*\tau_1 - R_{1r2s}/2 - R_{2r1s}/2

D_m Pb_n = A_mn - i B_mn
A_mn = -R_{1m1n}*\tau_2 + R_{1m1n}*(\tau_1**2 + \tau_2**2)/(2*\tau_2) + R_{2m2n}/(2*\tau_2) - \tau_1*(R_{1m2n} + R_{2m1n})/(2*\tau_2)
B_mn = R_{1m1n}*\tau_1 - R_{1m2n}/2 - R_{2m1n}/2


In [19]:
T_mnrs = sp.expand(sp.simplify((A_mn - sp.I * B_mn) * (A_rs + sp.I * B_rs)))
# remove imagainary components because they don't contribute
T_mnrs = sp.re(sp.expand(T_mnrs))
T_mnrs = T_mnrs.replace(lambda e: e.func == sp.re, lambda e: e.args[0])
T_mnrs = T_mnrs.replace(lambda e: e.func == sp.im, lambda e: sp.S.Zero)

In [20]:
T_mnrs

R_{1m1n}*R_{1r1s}*\tau_1**4/(4*\tau_2**2) + R_{1m1n}*R_{1r1s}*\tau_1**2/2 + R_{1m1n}*R_{1r1s}*\tau_2**2/4 - R_{1m1n}*R_{1r2s}*\tau_1**3/(4*\tau_2**2) - R_{1m1n}*R_{1r2s}*\tau_1/4 - R_{1m1n}*R_{2r1s}*\tau_1**3/(4*\tau_2**2) - R_{1m1n}*R_{2r1s}*\tau_1/4 + R_{1m1n}*R_{2r2s}*\tau_1**2/(4*\tau_2**2) - R_{1m1n}*R_{2r2s}/4 - R_{1m2n}*R_{1r1s}*\tau_1**3/(4*\tau_2**2) - R_{1m2n}*R_{1r1s}*\tau_1/4 + R_{1m2n}*R_{1r2s}*\tau_1**2/(4*\tau_2**2) + R_{1m2n}*R_{1r2s}/4 + R_{1m2n}*R_{2r1s}*\tau_1**2/(4*\tau_2**2) + R_{1m2n}*R_{2r1s}/4 - R_{1m2n}*R_{2r2s}*\tau_1/(4*\tau_2**2) - R_{1r1s}*R_{2m1n}*\tau_1**3/(4*\tau_2**2) - R_{1r1s}*R_{2m1n}*\tau_1/4 + R_{1r1s}*R_{2m2n}*\tau_1**2/(4*\tau_2**2) - R_{1r1s}*R_{2m2n}/4 + R_{1r2s}*R_{2m1n}*\tau_1**2/(4*\tau_2**2) + R_{1r2s}*R_{2m1n}/4 - R_{1r2s}*R_{2m2n}*\tau_1/(4*\tau_2**2) + R_{2m1n}*R_{2r1s}*\tau_1**2/(4*\tau_2**2) + R_{2m1n}*R_{2r1s}/4 - R_{2m1n}*R_{2r2s}*\tau_1/(4*\tau_2**2) - R_{2m2n}*R_{2r1s}*\tau_1/(4*\tau_2**2) + R_{2m2n}*R_{2r2s}/(4*\tau_2**2)

In [21]:
# Raise all the indices of R_ambn etnsors
R_1m1n_up = sp.Symbol(r'R^{1}{}_{m}{}^{1}{}_{n}')
R_1m2n_up = sp.Symbol(r'R^{1}{}_{m}{}^{2}{}_{n}')
R_2m1n_up = sp.Symbol(r'R^{2}{}_{m}{}^{1}{}_{n}')
R_2m2n_up = sp.Symbol(r'R^{2}{}_{m}{}^{2}{}_{n}')

g_11 = 1 / tau2_sym
g_12 = tau1_sym / tau2_sym
g_22 = (tau1_sym**2 + tau2_sym**2)/tau2_sym

ginv_11 = g_22
ginv_12 = -g_12
ginv_22 = g_11

# R_{ambn} = g_{ac} g_{bd} R^c_m^d_n
#R_1m1n_sub = (R_1m1n, g_11**2 * R_1m1n_up + g_11 * g_12 * (R_1m2n_up + R_2m1n_up) + g_12**2 * R_2m2n_up)
# R_1m2n_sub = (R_1m2n, g_11 * g_12 * R_1m1n_up + g_11 * g_22 * R_1m2n_up + g_12**2 * R_2m1n_up + g_12 * g_22 * R_2m2n_up)
# R_2m1n_sub = (R_2m1n, g_11 * g_12 * R_1m1n_up + g_12**2 * R_1m2n_up + g_11 * g_22 * R_2m1n_up + g_12 * g_22 * R_2m2n_up)
#R_2m2n_sub = (R_2m2n, g_12**2 * R_1m1n_up + g_12 * g_22 * (R_1m2n_up + R_2m1n_up) + g_22**2 * R_2m2n_up)

# T_mnrs = T_mnrs.subs([R_1m2n_sub, R_2m1n_sub])
# T_mnrs = sp.expand(sp.simplify(T_mnrs))

In [38]:
ginv_11_sym = sp.Symbol(r'g^{11}')
ginv_12_sym = sp.Symbol(r'g^{12}')
ginv_22_sym = sp.Symbol(r'g^{22}')

epsilon_12_up = sp.Symbol(r'\epsilon^{12}')

simplification_subs = {
    tau1_sym**4 / tau2_sym**2:   (ginv_11_sym**2 - (ginv_11 **2 - tau1_sym**4 / tau2_sym**2)),
    - tau1_sym**3 / tau2_sym**2:   (ginv_11_sym*ginv_12_sym - (ginv_11 *ginv_12 + tau1_sym**3 / tau2_sym**2)),
    (R_1r2s) * tau1_sym**2 / tau2_sym**2:  (R_1r2s) * (ginv_11_sym*ginv_22_sym-1),
    (R_2r1s) * tau1_sym**2 / tau2_sym**2:  (R_2r1s) * (ginv_11_sym*ginv_22_sym-1),
    (R_2r2s) * tau1_sym**2 / tau2_sym**2:  (R_2r2s) * (ginv_12_sym**2),
    (R_2m2n) * tau1_sym**2 / tau2_sym**2:  (R_2m2n) * (ginv_12_sym**2),
    (R_2r2s) * (- tau1_sym / tau2_sym**2): (R_2r2s) * (ginv_12_sym * ginv_22_sym),
    (R_2m2n) * (- tau1_sym / tau2_sym**2): (R_2m2n) * (ginv_12_sym * ginv_22_sym),
    (R_2m2n) * 1/ tau2_sym **2 : (R_2m2n) * ginv_22_sym**2,
    (R_2r2s) * sp.Rational(-1, 4): sp.Rational(-1, 4) * (R_2r2s) * epsilon_12_up**2,
    (R_2m2n) * sp.Rational(-1, 4): sp.Rational(-1, 4) * (R_2m2n) * epsilon_12_up**2,
    # (R_2m2n) * (- tau1_sym / tau2_sym**2): (R_2m2n) * (ginv_12_sym * ginv_22_sym),
    
    #- tau1_sym / tau2_sym**2: ginv_12_sym * ginv_22_sym,
}
T_mnrs = T_mnrs.subs(simplification_subs)
T_mnrs = sp.expand(sp.simplify(T_mnrs))

In [42]:
sp.collect(T_mnrs, [ginv_11_sym ** 2, ginv_11_sym * ginv_12_sym, ginv_12_sym**2, ginv_12_sym * ginv_22_sym, ginv_22_sym**2, epsilon_12_up**2]) 

R_{1m1n}*R_{1r1s}*g^{11}**2/4 - R_{1m1n}*R_{2r2s}*\epsilon^{12}**2/4 + R_{1m2n}*R_{1r2s}*g^{11}*g^{22}/4 + R_{1m2n}*R_{2r1s}*g^{11}*g^{22}/4 - R_{1r1s}*R_{2m2n}*\epsilon^{12}**4/4 + R_{1r2s}*R_{2m1n}*g^{11}*g^{22}/4 + R_{2m1n}*R_{2r1s}*g^{11}*g^{22}/4 + R_{2m2n}*R_{2r2s}*g^{22}**2/4 + g^{11}*g^{12}*(R_{1m1n}*R_{1r2s}/4 + R_{1m1n}*R_{2r1s}/4 + R_{1m2n}*R_{1r1s}/4 + R_{1r1s}*R_{2m1n}/4) + g^{12}**2*(R_{1m1n}*R_{2r2s}/4 + R_{1r1s}*R_{2m2n}/4) + g^{12}*g^{22}*(R_{1m2n}*R_{2r2s}/4 + R_{1r2s}*R_{2m2n}/4 + R_{2m1n}*R_{2r2s}/4 + R_{2m2n}*R_{2r1s}/4)

Which, in Einstein summation form, is

$$
D_m\bar{P}_n D_r P_s = \frac{1}{4}(g^{ab}g^{cd} - \epsilon^{ab}\epsilon^{cd})R_{amcn}R_{brds}
$$

So that for the four-index tensor $(DP)_{mpnq} \equiv g_{[m[n} D_{p]} P_{q]}$ antisymmetrized in $mp$, and $nq$ with overall normalization 1. 
We have (note that $D_p P_q$ by definition is symmetric in $pq$)

$$
|DP|^2_{m'p'n'q' mpnq} = g_{m'n'}g_{mn} \frac{1}{4}(g^{ab}g^{cd} - \epsilon^{ab}\epsilon^{cd})R_{a p' c q'}R_{b p d q}
$$

antisymmetrized in $[m'p']$, $[n'q']$, $[mp]$, $[nq]$.